## Import Required Libraries

In [1]:
%pip list

Package                 Version
----------------------- -----------
asttokens               3.0.2
colorama                0.4.6
comm                    0.2.3
contourpy               1.3.3
cycler                  0.12.1
debugpy                 1.8.21
duckdb                  1.5.5
executing               2.2.1
fonttools               4.63.0
ipykernel               7.3.0
ipython                 9.16.1
ipython_pygments_lexers 1.1.1
jedi                    0.20.0
joblib                  1.5.3
jupyter_client          8.9.1
jupyter_core            5.9.1
kiwisolver              1.5.0
matplotlib              3.11.1
matplotlib-inline       0.2.2
narwhals                2.24.0
nest-asyncio2           1.7.2
numpy                   2.5.1
packaging               26.3
pandas                  3.0.5
parso                   0.8.7
pillow                  12.3.0
pip                     26.2.1
platformdirs            4.11.1
prompt_toolkit          3.0.53
psutil                  7.2.2
pure_eval             

In [ ]:
# Core
import numpy as np
import pandas as pd

# Scikit-learn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (StandardScaler, OneHotEncoder)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# XGBoost
from xgboost import XGBClassifier

# Model persistence
import joblib

Matplotlib is building the font cache; this may take a moment.


## Load the Modeling Splits

In [3]:
# Load modeling datasets
train = pd.read_parquet("../data/processed/modeling/train.parquet")
validation = pd.read_parquet("../data/processed/modeling/validation.parquet")
test = pd.read_parquet("../data/processed/modeling/test.parquet")

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (2100009, 9)
Validation shape: (449656, 9)
Test shape: (450335, 9)


## Verify Target Distribution

In [4]:
# Verify target distribution across splits
for name, df in [("Train", train), ("Validation", validation), ("Test", test)]:
    fraud_rate = df["is_fraud"].mean()

    print(f"{name}:")
    print(f"  Fraud cases: {df['is_fraud'].sum():,}")
    print(f"  Legitimate cases: {(df['is_fraud'] == 0).sum():,}")
    print(f"  Fraud rate: {fraud_rate:.4%}")
    print()

Train:
  Fraud cases: 31,589
  Legitimate cases: 2,068,420
  Fraud rate: 1.5042%

Validation:
  Fraud cases: 6,653
  Legitimate cases: 443,003
  Fraud rate: 1.4796%

Test:
  Fraud cases: 6,758
  Legitimate cases: 443,577
  Fraud rate: 1.5007%



## Separate Features and Target

In [5]:
# Separate features and target
target = "is_fraud"

X_train = train.drop(columns=target)
y_train = train[target]

X_validation = validation.drop(columns=target)
y_validation = validation[target]

X_test = test.drop(columns=target)
y_test = test[target]

X_train.columns

Index(['amount', 'billing_country', 'route', 'card_bin', 'account_age_days',
       'currency', 'transaction_hour', 'transaction_dayofweek'],
      dtype='str')

## Define Feature Types

In [6]:
numeric_features = [
    "amount",
    "account_age_days",
    "transaction_hour",
    "transaction_dayofweek"
]

categorical_features = [
    "billing_country",
    "route",
    "card_bin",
    "currency"
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['amount', 'account_age_days', 'transaction_hour', 'transaction_dayofweek']
Categorical features: ['billing_country', 'route', 'card_bin', 'currency']


For Logistic Regression, scaling the numerical variables and one-hot encoding the categorical variables would be fine. Since one-hot encoding produces a sparse matrix, we'll use `StandardScaler(with_mean=False)` so we don't unnecessarily convert the whole representation to dense.

We'll also use `OneHotEncoder(handle_unknown="ignore")` because a category can appear in validation/test that wasn't present in training. The pipeline should handle that gracefully rather than fail.

## Build Baseline Preprocessing

In [8]:
# Preprocessing for Logistic Regression
numeric_transformer = Pipeline(
    steps=[("scaler", StandardScaler(with_mean=False))]
)

categorical_transformer = Pipeline(
    steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


## Build the Logistic Regression Baseline

In [9]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                solver="saga",
                max_iter=100,
                random_state=42
            )
        )
    ]
)

print("Baseline model created successfully.")

Baseline model created successfully.


- `class_weight="balanced"` gives the minority fraud class greater influence during training.
- `solver="saga"` is suitable for large datasets and sparse feature matrices.
- `max_iter=100` is reasonable initial limit. We'll see whether convergence is actually achieved rather than blindly setting a huge number.
- `random_state=42` for reproducibility.

## Train Logistic Regression

In [ ]:
baseline_model.fit(X_train, y_train)
print("Baseline model training completed.")

### Check Logistic Regression convergence status

In [11]:
classifier = baseline_model.named_steps["classifier"]

print("Iterations used:", classifier.n_iter_[0])
print("Maximum iterations:", classifier.max_iter)
print("Converged:", classifier.n_iter_[0] < classifier.max_iter)

Iterations used: 100
Maximum iterations: 100
Converged: False


## Refine numerical preprocessing for Logistic Regression

In [12]:
numeric_transformer = Pipeline(
    steps=[("scaler", StandardScaler())]
)

categorical_transformer = Pipeline(
    steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                solver="saga",
                max_iter=200,
                random_state=42
            )
        )
    ]
)

print("Baseline preprocessing and model updated.")

Baseline preprocessing and model updated.


### Inspect Categorical Cardinality

In [13]:
for col in categorical_features:
    print(f"{col}: {X_train[col].nunique():,} unique values")

billing_country: 4 unique values
route: 1,558 unique values
card_bin: 24 unique values
currency: 7 unique values


### Adjust Baseline Convergence Settings

In [14]:
baseline_model.set_params(classifier__tol=1e-3)

print("Updated Logistic Regression settings:")
print("  Solver:", baseline_model.named_steps["classifier"].solver)
print("  Class weight:", baseline_model.named_steps["classifier"].class_weight)
print("  Max iterations:", baseline_model.named_steps["classifier"].max_iter)
print("  Tolerance:", baseline_model.named_steps["classifier"].tol)

Updated Logistic Regression settings:
  Solver: saga
  Class weight: balanced
  Max iterations: 200
  Tolerance: 0.001


In [15]:
baseline_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['amount','billing_country','route',...,'currency','transaction_hour', 'transaction_dayofweek']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remai

### Verify Baseline Convergence

In [16]:
classifier = baseline_model.named_steps["classifier"]

print("Iterations used:", classifier.n_iter_[0])
print("Maximum iterations:", classifier.max_iter)
print("Converged:", classifier.n_iter_[0] < classifier.max_iter)

Iterations used: 169
Maximum iterations: 200
Converged: True


## Generate Validation Predictions

In [17]:
y_validation_proba = baseline_model.predict_proba(X_validation)[:, 1]
y_validation_pred = baseline_model.predict(X_validation)

print("Probability predictions:", y_validation_proba.shape)
print("Class predictions:", y_validation_pred.shape)

Probability predictions: (449656,)
Class predictions: (449656,)


## Evaluate Logistic Regression: Default Threshold

In [18]:
# Evaluate Logistic Regression on the validation set
roc_auc = roc_auc_score(y_validation, y_validation_proba)
pr_auc = average_precision_score(y_validation, y_validation_proba)

precision = precision_score(y_validation, y_validation_pred)
recall = recall_score(y_validation, y_validation_pred)
f1 = f1_score(y_validation, y_validation_pred)

print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

ROC-AUC : 0.5935
PR-AUC  : 0.0764
Precision: 0.0237
Recall   : 0.5028
F1-score : 0.0453


At first glance, it looks pretty bad. And honestly, it is a weak baseline.

But let's not misread `0.5935` as "the model is 59.35% accurate". ROC-AUC is not accuracy.

ROC-AUC = 0.5935 means: very roughly, if we randomly take one fraudulent transaction and one legitimate transaction, the model has about a `59.35%` chance of assigning the fraudulent transaction a higher fraud score.

Our Logistic Regression is therefore only slightly better than random ranking. That's exactly why we needed a baseline. If we had started with XGBoost and gotten, say, 0.8 PR-AUC without establishing this, we'd know the result... but we wouldn't know how much value the nonlinear model actually added.

PR-AUC is even more revealing. Our validation fraud prevalence is 1.4796%. So a random classifier's expected PR-AUC is roughly around 0.0148.

Our Logistic Regression gets 0.0764. That's substantially above the baseline prevalence, so the model does contain some useful ranking signal.

The baseline has weak overall discrimination, but it extracts some useful fraud-ranking signal from the available features.

## Baseline Confusion Matrix

In [20]:
cm = confusion_matrix(y_validation, y_validation_pred)

print("Confusion Matrix:\n", cm)

tn, fp, fn, tp = cm.ravel()

print(f"\nTrue Negatives : {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives : {tp:,}")

Confusion Matrix:
 [[305367 137636]
 [  3308   3345]]

True Negatives : 305,367
False Positives: 137,636
False Negatives: 3,308
True Positives : 3,345


### Inspect Validation Fraud Probabilities

In [ ]:
print("Minimum probability:", y_validation_proba.min())
print("Maximum probability:", y_validation_proba.max())
print("Mean probability   :", y_validation_proba.mean())
print("Median probability :", np.median(y_validation_proba))

print("\nSelected percentiles:")
for p in [90, 95, 97, 98, 99, 99.5, 99.9]:
    print(f"{p:>5.1f}th percentile: {np.percentile(y_validation_proba, p):.4f}")

Minimum probability: 0.017767232764933596
Maximum probability: 0.9956181629585539
Mean probability   : 0.46335419735292827
Median probability : 0.4555765534870383

Selected percentiles:
 90.0th percentile: 0.5897
 95.0th percentile: 0.6383
 97.0th percentile: 0.6704
 98.0th percentile: 0.6937
 99.0th percentile: 0.7316
 99.5th percentile: 0.7674
 99.9th percentile: 0.8424


## Compare Predicted Scores by Class

In [23]:
# Compare predicted fraud scores for legitimate and fraudulent transactions
fraud_scores = y_validation_proba[y_validation == 1]
legitimate_scores = y_validation_proba[y_validation == 0]

print("Fraud transactions:")
print(f"  Mean score:   {fraud_scores.mean():.4f}")
print(f"  Median score: {np.median(fraud_scores):.4f}")

print("\nLegitimate transactions:")
print(f"  Mean score:   {legitimate_scores.mean():.4f}")
print(f"  Median score: {np.median(legitimate_scores):.4f}")

Fraud transactions:
  Mean score:   0.5157
  Median score: 0.5012

Legitimate transactions:
  Mean score:   0.4626
  Median score: 0.4552


The model is separating the classes, but very weakly:
- Fraud mean = 0.5157
- Legitimate mean = 0.4626
- Difference ≈ 0.0531
And the medians are even closer:
- Fraud = 0.5012
- Legitimate = 0.4552
- Difference ≈ 0.0460

So the distributions are clearly overlapping heavily. That's consistent with our: `ROC-AUC = 0.5935`

In other words, Logistic Regression has learned some signal, but the eight features in a linear model don't separate fraud from legitimate transactions very strongly.

That's enough baseline diagnosis. So rather than squeezing more information out of Logistic Regression, let's move to the next question:

> Can a nonlinear tree model discover interactions/patterns that the linear baseline cannot?

## Build XGBoost Preprocessing

In [24]:
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                dtype=np.float32
            ),
            categorical_features
        )
    ],
    sparse_threshold=1.0
)

print("XGBoost preprocessor created successfully.")

XGBoost preprocessor created successfully.


## Fit XGBoost Preprocessor on Training Data

In [25]:
X_train_xgb = xgb_preprocessor.fit_transform(X_train)
X_validation_xgb = xgb_preprocessor.transform(X_validation)

print("Training matrix shape:", X_train_xgb.shape)
print("Validation matrix shape:", X_validation_xgb.shape)
print("Training matrix type:", type(X_train_xgb).__name__)

Training matrix shape: (2100009, 1597)
Validation matrix shape: (449656, 1597)
Training matrix type: csr_matrix


CSR (Compressed Sparse Row) is a way of storing a matrix when most of its values are zero.

Our transformed matrix has `2,100,009 rows × 1,597 columns`. That's about `3.35 billion` potential cells.

But look at what one-hot encoding does. With `route`, there are `1,558` possible categories, but each transaction belongs to only one route. So most of those `~1,597` columns are zero for any individual transaction.

Instead of physically storing `[0, 0, 0, 1, 0, 0, 0, 0, ...]`, CSR essentially stores the non-zero values and where they occur, rather than wasting memory storing millions/billions of zeros.

Suppose we have this matrix:
```
[ 0  0  5  0  0 ]
[ 0  0  0  0  8 ]
[ 2  0  0  0  0 ]
[ 0  0  0  7  0 ]
```
CSR saves memory because it doesn't store the zeros at all. CSR instead stores three arrays.

1. **data**
    - The actual non-zero values: `data = [5, 8, 2, 7]`
2. **indices**
    - The column position of each non-zero value: `indices = [2, 4, 0, 3]`
3. **`indptr`**
    - This tells us where each row starts and ends inside data: `indptr = [0, 1, 2, 3, 4]`

Think of it as:
```
Row 0 → data[0:1] → [5]
Row 1 → data[1:2] → [8]
Row 2 → data[2:3] → [2]
Row 3 → data[3:4] → [7]
```
So CSR effectively says: "I have these 4 values, here are their columns, and here's where each row's values begin/end." It never needs to write down the 16 zeros.

Now let's actually see how sparse our matrix is rather than just saying "it's sparse", let's quantify it.

## Measure XGBoost Matrix Sparsity

In [26]:
# Measure sparsity of the transformed training matrix
total_elements = X_train_xgb.shape[0] * X_train_xgb.shape[1]
non_zero_elements = X_train_xgb.nnz
sparsity = 1 - (non_zero_elements / total_elements)

print(f"Total elements:     {total_elements:,}")
print(f"Non-zero elements:  {non_zero_elements:,}")
print(f"Zero elements:      {total_elements - non_zero_elements:,}")
print(f"Sparsity:           {sparsity:.2%}")

Total elements:     3,353,714,373
Non-zero elements:  16,411,508
Zero elements:      3,337,302,865
Sparsity:           99.51%


## Define Initial XGBoost Model

In [27]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

print("XGBoost model created successfully.")
print(f"scale_pos_weight: {xgb_model.scale_pos_weight:.2f}")

XGBoost model created successfully.
scale_pos_weight: 65.48


We're establishing a sensible starting point, not claiming these are optimal:
* `n_estimators=300` ⇰ Enough trees to learn meaningful patterns without immediately going extreme.
* `max_depth=6` ⇰ A depth of 6 means a tree can make roughly 6 levels of decisions from root to leaf. Higher depth can capture complicated relationships but increases the risk of overfitting.
    - max_depth=3  → simpler trees
    - max_depth=6  → moderately complex
    - max_depth=10 → very complex
* `learning_rate=0.1` ⇰ A smaller learning rate generally means you need more trees. For example:
    - learning_rate = 0.1 → perhaps 300 trees
    - learning_rate = 0.01 → perhaps 1000+ trees
* `subsample=0.8` ⇰ For each tree, randomly use 80% of the training rows.
* `colsample_bytree=0.8` ⇰ For every tree, XGBoost randomly selects 80% of the features. This can reduce overfitting and make the individual trees less correlated. So:
    - subsample ⇰ sample ROWS
    - colsample_bytree ⇰ sample COLUMNS
* `objective="binary:logistic"` ⇰ This tells XGBoost: "I'm doing binary classification, and I want probability outputs."
* `eval_metric="aucpr"` ⇰ This tells XGBoost: When evaluating the model, use PR-AUC. `aucpr` = Area Under the Precision-Recall Curve.
* `scale_pos_weight: 65.48` ⇰ That tells XGBoost: Make mistakes on fraud roughly 65× more important than mistakes on legitimate transactions.
* `tree_method="hist"` ⇰ Important for efficient training on a dataset this large. This tells XGBoost to use the histogram-based tree-building algorithm.
* `n_jobs=-1` ⇰ use available CPU cores. This controls how many CPU cores XGBoost can use. `-1` means "use all available CPU cores".

The training ratio is approximately: $2,068,420 / 31,589 ≈ 65.48$

So XGBoost will give substantially greater weight to the fraud class.

## Train Initial XGBoost Model

In [28]:
xgb_model.fit(
    X_train_xgb,
    y_train,
    eval_set=[(X_validation_xgb, y_validation)],
    verbose=50
)

print("XGBoost training completed.")

[0]	validation_0-aucpr:0.37770
[50]	validation_0-aucpr:0.40648
[100]	validation_0-aucpr:0.42324
[150]	validation_0-aucpr:0.43284
[200]	validation_0-aucpr:0.44048
[250]	validation_0-aucpr:0.44612
[299]	validation_0-aucpr:0.45010
XGBoost training completed.


> Training matrix ⇰ XGBoost ⇰ 300 boosting rounds ⇰ Validation PR-AUC monitored
A couple of important things:
* **Training data** is used to fit the trees.
* **Validation data** is used only to monitor performance during this initial development run.
* We're monitoring **PR-AUC**, because that's more informative for this 1.5%-fraud problem than accuracy.
* `verbose=50` means we won't get 300 lines of output—just periodic progress.

## Evaluate XGBoost on Validation Set

In [29]:
# Generate XGBoost validation predictions
y_validation_xgb_proba = xgb_model.predict_proba(X_validation_xgb)[:, 1]
y_validation_xgb_pred = (y_validation_xgb_proba >= 0.5).astype(int)

xgb_roc_auc = roc_auc_score(y_validation, y_validation_xgb_proba)
xgb_pr_auc = average_precision_score(y_validation, y_validation_xgb_proba)

xgb_precision = precision_score(y_validation, y_validation_xgb_pred)
xgb_recall = recall_score(y_validation, y_validation_xgb_pred)
xgb_f1 = f1_score(y_validation, y_validation_xgb_pred)

print(f"ROC-AUC : {xgb_roc_auc:.4f}")
print(f"PR-AUC  : {xgb_pr_auc:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1-score : {xgb_f1:.4f}")

ROC-AUC : 0.8899
PR-AUC  : 0.4501
Precision: 0.1183
Recall   : 0.6962
F1-score : 0.2022


## Evaluate XGBoost Across Thresholds

In [30]:
thresholds = np.arange(0.05, 0.96, 0.05)

threshold_results = []

for threshold in thresholds:
    y_pred = (y_validation_xgb_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_validation, y_pred).ravel()

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_validation, y_pred, zero_division=0),
        "recall": recall_score(y_validation, y_pred, zero_division=0),
        "f1": f1_score(y_validation, y_pred, zero_division=0),
        "flagged": tp + fp,
        "false_positives": fp,
        "fraud_captured": tp
    })

threshold_results = pd.DataFrame(threshold_results)
threshold_results

,threshold,precision,recall,f1,flagged,false_positives,fraud_captured
0,0.05,0.014927,0.999248,0.029415,445362,438714,6648
1,0.10,0.016781,0.991282,0.033003,393009,386414,6595
2,0.15,0.019345,0.977754,0.037939,336268,329763,6505
3,0.20,0.023305,0.958816,0.045504,273717,267338,6379
4,0.25,0.030210,0.913423,0.058485,201160,195083,6077
5,0.30,0.042143,0.863971,0.080365,136394,130646,5748
6,0.35,0.056583,0.816023,0.105827,95948,90519,5429
7,0.40,0.074633,0.776642,0.136180,69232,64065,5167
8,0.45,0.096301,0.733804,0.170259,50695,45813,4882
9,0.50,0.118308,0.696227,0.202249,39152,34520,4632


### Inspect XGBoost Validation Learning Curve

In [ ]:
# Inspect validation PR-AUC across boosting rounds
eval_history = xgb_model.evals_result()

validation_aucpr = eval_history["validation_0"]["aucpr"]

best_iteration = int(np.argmax(validation_aucpr))
best_aucpr = validation_aucpr[best_iteration]

print(f"Best iteration: {best_iteration}")
print(f"Best validation PR-AUC: {best_aucpr:.4f}")
print(f"Final validation PR-AUC: {validation_aucpr[-1]:.4f}")
print(f"Improvement over last 50 rounds: {validation_aucpr[-1] - validation_aucpr[-51]:.4f}")

Best iteration: 299
Best validation PR-AUC: 0.4501
Final validation PR-AUC: 0.4501
Improvement over last 50 rounds: 0.0040


## Compare Logistic Regression vs XGBoost

In [33]:
model_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost"],
    "ROC-AUC": [roc_auc, xgb_roc_auc],
    "PR-AUC": [pr_auc, xgb_pr_auc],
    "Precision": [precision, xgb_precision],
    "Recall": [recall, xgb_recall],
    "F1": [f1, xgb_f1]
})

model_comparison.round(4)

,Model,ROC-AUC,PR-AUC,Precision,Recall,F1
0,Logistic Regression,0.5935,0.0764,0.0237,0.5028,0.0453
1,XGBoost,0.8899,0.4501,0.1183,0.6962,0.2022


## Train Extended XGBoost Candidate

In [34]:
# Train an extended XGBoost candidate with early stopping
xgb_model_extended = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    tree_method="hist",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

xgb_model_extended.fit(
    X_train_xgb,
    y_train,
    eval_set=[(X_validation_xgb, y_validation)],
    verbose=50
)

print("Extended XGBoost training completed.")

[0]	validation_0-aucpr:0.37770
[50]	validation_0-aucpr:0.40648
[100]	validation_0-aucpr:0.42324
[150]	validation_0-aucpr:0.43284
[200]	validation_0-aucpr:0.44048
[250]	validation_0-aucpr:0.44612
[300]	validation_0-aucpr:0.44997
[350]	validation_0-aucpr:0.45426
[400]	validation_0-aucpr:0.45688
[450]	validation_0-aucpr:0.45826
[499]	validation_0-aucpr:0.45978
Extended XGBoost training completed.


### Inspect Extended XGBoost Best Iteration

In [35]:
best_iteration_extended = xgb_model_extended.best_iteration
best_score_extended = xgb_model_extended.best_score

print(f"Best iteration: {best_iteration_extended}")
print(f"Best validation PR-AUC: {best_score_extended:.4f}")
print(f"Trees configured: {xgb_model_extended.n_estimators}")

Best iteration: 492
Best validation PR-AUC: 0.4599
Trees configured: 500


## Generate Final XGBoost Validation Predictions

In [36]:
y_validation_xgb_final_proba = (
    xgb_model_extended.predict_proba(X_validation_xgb)[:, 1]
)

print("Validation predictions:", y_validation_xgb_final_proba.shape)
print(
    "Best validation PR-AUC:",
    average_precision_score(y_validation, y_validation_xgb_final_proba)
)

Validation predictions: (449656,)
Best validation PR-AUC: 0.45995723742346306


## Final XGBoost Threshold Analysis

In [37]:
# Evaluate the final XGBoost candidate across classification thresholds
thresholds = np.arange(0.50, 0.991, 0.01)

threshold_results_final = []

for threshold in thresholds:
    y_pred = (y_validation_xgb_final_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        y_pred
    ).ravel()

    threshold_results_final.append({
        "threshold": threshold,
        "precision": precision_score(y_validation, y_pred, zero_division=0),
        "recall": recall_score(y_validation, y_pred, zero_division=0),
        "f1": f1_score(y_validation, y_pred, zero_division=0),
        "flagged": tp + fp,
        "false_positives": fp,
        "fraud_captured": tp
    })

threshold_results_final = pd.DataFrame(threshold_results_final)

threshold_results_final.round(4)

,threshold,precision,recall,f1,flagged,false_positives,fraud_captured
0,0.50,0.1238,0.7003,0.2104,37626,32967,4659
1,0.51,0.1282,0.6910,0.2163,35859,31262,4597
2,0.52,0.1333,0.6844,0.2231,34167,29614,4553
3,0.53,0.1386,0.6767,0.2300,32490,27988,4502
4,0.54,0.1443,0.6686,0.2374,30820,26372,4448
5,0.55,0.1503,0.6600,0.2448,29222,24831,4391
6,0.56,0.1563,0.6520,0.2522,27752,23414,4338
7,0.57,0.1627,0.6438,0.2598,26323,22040,4283
8,0.58,0.1701,0.6375,0.2685,24936,20695,4241
9,0.59,0.1771,0.6307,0.2765,23698,19502,4196


## Summarize Candidate Operating Points

In [39]:
candidate_thresholds = [0.80, 0.85, 0.90, 0.91, 0.93, 0.95]

candidate_points = (
    threshold_results_final[
        threshold_results_final["threshold"].round(2).isin(candidate_thresholds)
    ]
    .copy()
    .reset_index(drop=True)
)

candidate_points.round(4)

,threshold,precision,recall,f1,flagged,false_positives,fraud_captured
0,0.80,0.3816,0.4599,0.4171,8019,4959,3060
1,0.85,0.4758,0.4114,0.4412,5753,3016,2737
2,0.90,0.6126,0.3701,0.4614,4019,1557,2462
3,0.91,0.6421,0.3610,0.4622,3741,1339,2402
4,0.93,0.7147,0.3359,0.4571,3127,892,2235
5,0.95,0.7700,0.3150,0.4471,2722,626,2096


## Lock the Primary Operating Threshold

In [40]:
final_threshold = 0.91

selected_point = (
    threshold_results_final[
        threshold_results_final["threshold"].round(2) == final_threshold
    ]
    .iloc[0]
)

print(f"Primary operating threshold: {final_threshold:.2f}")
print(f"Validation precision: {selected_point['precision']:.4f}")
print(f"Validation recall: {selected_point['recall']:.4f}")
print(f"Validation F1: {selected_point['f1']:.4f}")
print(f"Validation transactions flagged: {selected_point['flagged']:,}")
print(f"Validation false positives: {selected_point['false_positives']:,}")
print(f"Validation fraud captured: {selected_point['fraud_captured']:,}")

Primary operating threshold: 0.91
Validation precision: 0.6421
Validation recall: 0.3610
Validation F1: 0.4622
Validation transactions flagged: 3,741.0
Validation false positives: 1,339.0
Validation fraud captured: 2,402.0


## Transform the Test Set

In [41]:
# Transform the untouched test set using the fitted XGBoost preprocessor

X_test_xgb = xgb_preprocessor.transform(X_test)

print("Test matrix shape:", X_test_xgb.shape)
print("Test matrix type:", type(X_test_xgb).__name__)

Test matrix shape: (450335, 1597)
Test matrix type: csr_matrix


## Generate Final Test Predictions

In [42]:
y_test_proba = xgb_model_extended.predict_proba(X_test_xgb)[:, 1]
y_test_pred = (y_test_proba >= final_threshold).astype(int)

print("Test predictions:", y_test_proba.shape)
print(f"Operating threshold: {final_threshold:.2f}")
print(f"Transactions flagged: {y_test_pred.sum():,}")

Test predictions: (450335,)
Operating threshold: 0.91
Transactions flagged: 4,002


## Final Test Evaluation

In [43]:
test_roc_auc = roc_auc_score(y_test, y_test_proba)
test_pr_auc = average_precision_score(y_test, y_test_proba)

test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

print(f"ROC-AUC : {test_roc_auc:.4f}")
print(f"PR-AUC  : {test_pr_auc:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1-score : {test_f1:.4f}")

ROC-AUC : 0.9067
PR-AUC  : 0.4915
Precision: 0.6319
Recall   : 0.3742
F1-score : 0.4701


## Compare Final Validation vs Test Performance

In [44]:
# Compare final XGBoost performance on validation and test sets

final_validation_roc_auc = roc_auc_score(
    y_validation,
    y_validation_xgb_final_proba
)

final_validation_pr_auc = average_precision_score(
    y_validation,
    y_validation_xgb_final_proba
)

final_performance_comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Precision @ 0.91",
        "Recall @ 0.91",
        "F1 @ 0.91"
    ],
    "Validation": [
        final_validation_roc_auc,
        final_validation_pr_auc,
        test_precision if False else 0,
        test_recall if False else 0,
        test_f1 if False else 0
    ],
    "Test": [
        test_roc_auc,
        test_pr_auc,
        test_precision,
        test_recall,
        test_f1
    ]
})

# Use the locked validation threshold results
validation_point = (
    threshold_results_final[
        threshold_results_final["threshold"].round(2) == final_threshold
    ]
    .iloc[0]
)

final_performance_comparison.loc[
    final_performance_comparison["Metric"] == "Precision @ 0.91",
    "Validation"
] = validation_point["precision"]

final_performance_comparison.loc[
    final_performance_comparison["Metric"] == "Recall @ 0.91",
    "Validation"
] = validation_point["recall"]

final_performance_comparison.loc[
    final_performance_comparison["Metric"] == "F1 @ 0.91",
    "Validation"
] = validation_point["f1"]

final_performance_comparison.round(4)

,Metric,Validation,Test
0,ROC-AUC,0.8936,0.9067
1,PR-AUC,0.4600,0.4915
2,Precision @ 0.91,0.6421,0.6319
3,Recall @ 0.91,0.3610,0.3742
4,F1 @ 0.91,0.4622,0.4701


## Final Test Confusion Matrix

In [45]:
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print(f"\nTrue Negatives : {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives : {tp:,}")

Confusion Matrix:
[[442104   1473]
 [  4229   2529]]

True Negatives : 442,104
False Positives: 1,473
False Negatives: 4,229
True Positives : 2,529


## Create Final Model Summary

In [46]:
# Final model summary

final_model_summary = {
    "model": "XGBoost",
    "n_estimators": xgb_model_extended.n_estimators,
    "best_iteration": xgb_model_extended.best_iteration,
    "max_depth": xgb_model_extended.max_depth,
    "learning_rate": xgb_model_extended.learning_rate,
    "subsample": xgb_model_extended.subsample,
    "colsample_bytree": xgb_model_extended.colsample_bytree,
    "scale_pos_weight": xgb_model_extended.scale_pos_weight,
    "operating_threshold": final_threshold,
    "validation_roc_auc": roc_auc_score(
        y_validation,
        y_validation_xgb_final_proba
    ),
    "validation_pr_auc": average_precision_score(
        y_validation,
        y_validation_xgb_final_proba
    ),
    "validation_precision": validation_point["precision"],
    "validation_recall": validation_point["recall"],
    "validation_f1": validation_point["f1"],
    "test_roc_auc": test_roc_auc,
    "test_pr_auc": test_pr_auc,
    "test_precision": test_precision,
    "test_recall": test_recall,
    "test_f1": test_f1
}

pd.Series(final_model_summary)

model                     XGBoost
n_estimators                  500
best_iteration                492
max_depth                       6
learning_rate                 0.1
subsample                     0.8
colsample_bytree              0.8
scale_pos_weight        65.479122
operating_threshold          0.91
validation_roc_auc        0.89361
validation_pr_auc        0.459957
validation_precision     0.642074
validation_recall         0.36104
validation_f1             0.46219
test_roc_auc             0.906721
test_pr_auc              0.491458
test_precision           0.631934
test_recall              0.374223
test_f1                  0.470074
dtype: object

## Save Final Modeling Artifact

In [ ]:
from pathlib import Path

In [48]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

final_model_bundle = {
    "preprocessor": xgb_preprocessor,
    "model": xgb_model_extended,
    "threshold": final_threshold,
    "metadata": final_model_summary
}

model_path = models_dir / "airline_fraud_xgboost.joblib"

joblib.dump(final_model_bundle, model_path)

print(f"Final model bundle saved to: {model_path}")
print(f"File size: {model_path.stat().st_size / (1024 ** 2):.2f} MB")

Final model bundle saved to: ..\models\airline_fraud_xgboost.joblib
File size: 1.70 MB


## Verify Saved Model Bundle

In [49]:
# Verify that the saved model bundle can be loaded and used

loaded_bundle = joblib.load(model_path)

loaded_preprocessor = loaded_bundle["preprocessor"]
loaded_model = loaded_bundle["model"]
loaded_threshold = loaded_bundle["threshold"]

test_sample = X_test.iloc[:5]

sample_transformed = loaded_preprocessor.transform(test_sample)
sample_scores = loaded_model.predict_proba(sample_transformed)[:, 1]
sample_predictions = (
    sample_scores >= loaded_threshold
).astype(int)

print("Bundle loaded successfully. ✅")
print(f"Loaded threshold: {loaded_threshold:.2f}")
print(f"Sample scores: {sample_scores}")
print(f"Sample predictions: {sample_predictions}")

Bundle loaded successfully. ✅
Loaded threshold: 0.91
Sample scores: [0.17132166 0.43099943 0.20938107 0.0799797  0.05576297]
Sample predictions: [0 0 0 0 0]


# Final Modeling Summary

The modeling workflow compared a Logistic Regression baseline with XGBoost using the training and validation sets, followed by final evaluation on the untouched test set.

### Model Selection

Logistic Regression provided a simple baseline but showed limited discriminatory power:

- Validation ROC-AUC: 0.5935
- Validation PR-AUC: 0.0764

XGBoost captured substantially stronger nonlinear patterns and interactions:

- Best validation PR-AUC: 0.4600
- Best iteration: 492
- Final configuration: 500 maximum boosting rounds

XGBoost was selected as the final model candidate.

### Threshold Selection

The default classification threshold was not treated as a business decision.

Threshold analysis on the validation set showed a clear precision-recall trade-off. A threshold of **0.91** was selected as the primary technical operating point because it provided the highest F1-score in the evaluated threshold grid while keeping the number of flagged transactions relatively low.

This threshold is a technical operating reference, not a universally optimal business threshold. Actual production policy should consider the relative costs of missed fraud and unnecessary customer intervention.

### Final Test Performance

On the untouched test set at the locked threshold of 0.91:

- ROC-AUC: **0.9067**
- PR-AUC: **0.4915**
- Precision: **63.19%**
- Recall: **37.42%**
- F1-score: **47.01%**

Confusion matrix:

- True Negatives: **442,104**
- False Positives: **1,473**
- False Negatives: **4,229**
- True Positives: **2,529**

The validation and test results were broadly consistent, supporting reasonable generalization of the selected model and operating threshold.

### Final Artifact

The fitted XGBoost model, preprocessing object, operating threshold, and model metadata were saved as:

`models/airline_fraud_xgboost.joblib`

The artifact was reloaded successfully and verified on sample transactions.